# Práctica 6: Fórmula de Black-Scholes

## Ejercicio

Considera un modelo lognormal de precios con $S_{0}=100$, $\sigma=20\%$ anual, $r=5\%$, con composición continua con un horizonte temporal $T=1$ año. Calcula el precio de una opción de compra $C$ con valor strike $K=110$ y el valor de la opción de venta $P$ con el mismo strike. Comprueba la fórmula de la paridad Put-Call:
$$C_{0}-P_{0}=F_{0}$$
donde $F_{0}$ es el precio del contrato forward con strike $K$ y vencimiento $T$. Compara con el precio obtenido en el ejercicio para el modelo Browniano Geométrico discreto.

In [ ]:
import numpy as np
from scipy.stats import norm
import matplotlib.pyplot as plt
# definimos los parámetros
S0 = 100
sigma = 0.2
K = 110
r = 0.05
T = 1

In [ ]:
Phi = norm.cdf # la distribución normal acumulada


def B(t):
    return np.exp(r*t)

def d1(S,t):
    return np.log(S*B(T-t)/K)/(sigma*np.sqrt(T-t))+sigma*np.sqrt(T-t)/2

def d2(S,t):
    return np.log(S*B(T-t)/K)/(sigma*np.sqrt(T-t))-sigma*np.sqrt(T-t)/2

def C(S,t):
    return 1/B(T-t)*(S*B(T-t)*Phi(d1(S,t))-K*Phi(d2(S,t)))

def P(S,t):
    return 1/B(T-t)*(K*Phi(-d2(S,t))-S*B(T-t)*Phi(-d1(S,t)))

print(C(S0,0)) # precio Call
print(P(S0,0)) # precio Put
print(C(S0,0)-P(S0,0)) # paridad put-call
print(S0 - K*np.exp(-r*T)) # verificación paridad Put-Call

## Práctica

Realiza la siguiente simulación con paso temporal $\tau=T/100$:

- Considera un activo S que sigue un movimiento Browniano Geométrico como en el ejercicio anterior, y con deriva $\mu=0.15$ y $p=1/2$.
- Dado un camino $\left\{ S_{n}\right\} _{n=0,N}$ simulado en la medida de probabilidad p (es decir, en el mundo real), calcula los valores de la cartera de replicación para la opción de compra del apartado anterior, usando las fórmulas para el modelo continuo.
- Compara el valor final (es decir, a tiempo $T$) de lo que paga la opción para el valor de $S_{T}$ en ese camino, y el de la cartera de replicación por ese mismo camino.
- Simula el procedimiento anterior 1000 veces, guardando en cada caso la diferencia entre $C\left(S_{T},T\right)$ y la cartera de replicación $V_{T}$. Dibuja un histograma con estas diferencias.

## Indicaciones para la práctica

### Inicialización:
Comenzamos con un valor de $S_{0}$ y, junto a los parámetros $K,T$ de la opción y el tipo de interés $r$ del mercado, calculamos su precio inicial $C_{0}$ con la fórmula de Black. Calculamos también $\Delta_{0}=\Phi\left(d_{1}(S_0)\right)$ para determinar el número de acciones a comprar. Formamos una cartera $V$ con la estructura:  
$$V_{t}=\left(C_{0}-\Delta_{0}S_{0}\right)e^{rt}+\Delta_{0}S_{t}\quad t\in[0,\tau]\\
V_{1}\equiv V_{t_{1}}=\left(V_{0}-\Delta_{0}S_{0}\right)e^{r\tau}+\Delta_{0}S_{1}$$

Nótese que $V$ está construida de tal manera que $V_{0}\equiv C_{0}$.

### Un paso temporal
Asumiendo conocido el valor de la cartera después de $n$ pasos ($V_{n}$), calculamos $\Delta_{n}=\Phi\left(d_1(S_{n})\right)$ y construimos la cartera para el intervalo $[t_{n},t_{n+1}]$:
$$
V_{t}=\left(V_{n}-\Delta_{n}S_{n}\right)e^{r\left(t-t_{n}\right)}+\Delta_{n}S_t\quad t\in[t_{n},t_{n+1}]\\
V_{n+1}=\left(V_{n}-\Delta_{n}S_{n}\right)e^{r\tau}+\Delta_{n}S_{n+1}
$$

### Desarrollo
Con este esquema obtenemos una cartera autofinanciada, que utiliza el dinero disponible en el instante $n$ para ajustar sus parámetros. Al llegar al instante $t_{N}\equiv T$ el valor de la cartera debe coincidir con el de la opción C europea, salvo un pequeño error ocasionado por el paso temporal finito.
La cartera se construye paso a paso de manera que replique al instrumento en cada paso temporal salvo un error que tiende a cero con $\tau$.

